# 🐍 Python from Scratch — Module 8: Text Processing — Regular Expressions and CSV

### When `.split(",")` isn't enough anymore

In Module 5, you parsed a CSV file by hand, using `.split(",")` — it worked, but it
breaks easily (e.g. a comma inside a value throws everything off). This module covers
two tools for more serious text work: regular expressions (`re`) for pattern matching,
and the `csv` module for properly handling CSV files.

## Table of Contents

1. [String methods — a quick reminder](#sec1)
2. [Regular expressions — the basics](#sec2)
3. [Patterns — character classes and quantifiers](#sec3)
4. [Groups and `re.sub()`](#sec4)
5. [The `csv` module — why not just `.split(",")`](#sec5)
6. [`csv.DictReader` and `csv.DictWriter`](#sec6)
7. [Fun fact: when NOT to use regex](#sec7)
8. [Module summary](#sec8)
9. [Exercises](#sec9)

---

<a id="sec1"></a>
## 1. String methods — a quick reminder

Before reaching for regex, check whether plain string methods already do the job —
they're faster and more readable for simple cases.

In [ ]:
text = "  Python is awesome!  "

print(text.strip())              # removes leading/trailing whitespace
print(text.strip().lower())      # lowercase
print(text.strip().replace("awesome", "great"))
print(text.strip().split(" "))   # split into a list of words
print("-".join(["a", "b", "c"])) # join a list into a string
print("Python" in text)          # membership check
print(text.strip().startswith("Python"))

<a id="sec2"></a>
## 2. Regular expressions — the basics

A **regular expression** (regex) is a pattern describing what a piece of text should
look like — much more powerful than `.find()`, since it can search for e.g. "a run of
digits" rather than one specific value. Python's `re` module handles them.

In [ ]:
import re

text = "Call 123-456-789 or email test@example.com"

# re.search() - finds the FIRST match anywhere in the text
match = re.search(r"\d{3}-\d{3}-\d{3}", text)
print(match.group())   # the whole matched piece

# re.findall() - returns a list of ALL matches
emails = re.findall(r"\S+@\S+\.\S+", text)
print(emails)

# re.sub() - replaces matches with different text
censored = re.sub(r"\d{3}-\d{3}-\d{3}", "[HIDDEN]", text)
print(censored)

> ⚠️ **The `r` prefix before a pattern**
>
> Regex patterns are almost always written as *raw strings* - with an `r` before the quote (`r"\d+"`). Without it, Python would try to interpret `\d` as an escape sequence (like `\n`), which breaks the pattern. `r"..."` tells Python: "don't process any backslashes here, pass them to the regex engine literally."

<a id="sec3"></a>
## 3. Patterns — character classes and quantifiers

The key building blocks of a pattern:

| Symbol | Meaning |
|---|---|
| `\d` | any digit (0-9) |
| `\w` | any "word" character (letter, digit, `_`) |
| `\s` | any whitespace character (space, tab, newline) |
| `.` | any character (except newline) |
| `*` | the previous element: 0 or more times |
| `+` | the previous element: 1 or more times |
| `?` | the previous element: 0 or 1 time (optional) |
| `{n,m}` | the previous element: between `n` and `m` times |
| `^` / `$` | start / end of the text (or line) |

In [ ]:
import re

texts = ["cat123", "dog", "5 cats", "bird99xyz"]

for t in texts:
    contains_digit = re.search(r"\d+", t) is not None   # contains at least one digit
    print(f"{t}: {contains_digit}")

# Matching the WHOLE string (^ and $) - digits only, nothing else:
for t in ["12345", "123a5", "abc"]:
    is_all_digits = re.match(r"^\d+$", t) is not None
    print(f"{t}: {is_all_digits}")

<a id="sec4"></a>
## 4. Groups and `re.sub()`

Parentheses `(...)` in a pattern create a **group** — a piece of the match you can refer
to separately afterward, e.g. to pull out just one part of the matched text.

In [ ]:
import re

date_text = "The meeting is on 15-03-2026"
match = re.search(r"(\d{2})-(\d{2})-(\d{4})", date_text)

print(match.group())    # the whole match: 15-03-2026
print(match.group(1))   # first group: day -> 15
print(match.group(2))   # second group: month -> 03
print(match.group(3))   # third group: year -> 2026

# re.sub() with groups - swap DD-MM-YYYY format for YYYY/MM/DD:
new_format = re.sub(r"(\d{2})-(\d{2})-(\d{4})", r"\3/\2/\1", date_text)
print(new_format)

<a id="sec5"></a>
## 5. The `csv` module — why not just `.split(",")`

In Module 5 you split CSV lines with `.split(",")`. The problem: what if a value
**itself contains a comma** (e.g. `"Smith, John",25`)? A manual split falls apart. The
built-in `csv` module correctly handles quoting, commas inside values, and other messy
edge cases.

In [ ]:
import csv

# Let's create a CSV file with a header and a comma INSIDE a value:
with open("people.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["full_name", "age", "city"])
    writer.writerow(["John Smith, Jr.", 25, "Warsaw"])
    writer.writerow(["Ania Nowak", 30, "Krakow"])

# Reading it back - csv.reader correctly knows the comma in "Jr." is part of the value:
with open("people.csv", "r", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    for row in reader:
        print(row)

> ⚠️ **The `newline=""` parameter**
>
> When working with CSV files in Python, always open the file with `newline=""` (an empty string) - without it, on Windows you can get extra blank rows between each record, because the OS and the `csv` module interpret line endings differently.

<a id="sec6"></a>
## 6. `csv.DictReader` and `csv.DictWriter`

Instead of lists (where you have to remember that index `0` is the name and `1` is the
age), `DictReader`/`DictWriter` work with dictionaries keyed by column name — clearer
and safer.

In [ ]:
import csv

with open("people.csv", "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        print(row)                # this is a regular dictionary
        print(row["full_name"])   # access by column name, not by index

# Writing from dictionaries:
data = [
    {"product": "Bread", "price": 4.5},
    {"product": "Milk", "price": 3.2},
]

with open("products.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["product", "price"])
    writer.writeheader()
    writer.writerows(data)

with open("products.csv", "r", encoding="utf-8") as file:
    print(file.read())

<a id="sec7"></a>
## 7. Fun fact: when NOT to use regex

Regex is powerful, but easy to overuse.

> 💡 **Fun fact**
>
> There's a well-known programmer's joke: "Some people, when confronted with a problem, think 'I know, I'll use regular expressions.' Now they have two problems." Regex can get unreadable, hard to debug, and it's easy to miss an edge case. For simple things (does a string start with something, split on a comma), plain string methods are the better choice - save regex for cases where the pattern is genuinely variable (e.g. "any number of digits", "an email in roughly any format").

<a id="sec8"></a>
## 8. Module summary

By now it should be clear:

- when plain string methods are enough, and when regex is worth reaching for,
- the basic pattern symbols (`\d`, `\w`, `\s`, `*`, `+`, `?`, `{n,m}`),
- the difference between `re.search()`, `re.match()`, `re.findall()`, and `re.sub()`,
- how groups work in a regex,
- why the `csv` module is safer than a manual `.split(",")`,
- how to use `csv.DictReader`/`csv.DictWriter` to work with dictionaries instead of
  lists.

Next module: **testing your code** — how to check that your functions actually work
correctly, automatically and repeatably, instead of eyeballing `print()` output by
hand.

<a id="sec9"></a>
## 9. Exercises

A few tasks combine regex with lists/loops, and the rest is practical CSV work,
including a proper fix for the Module 5 approach.

> 📝 **Exercise 1: Does this look like an email**
>
> Write a function `is_email(text)` that checks (with a simplified regex, it doesn't need to be 100% formally correct) whether a string looks like an email address - something, then `@`, then something, a dot, then something. Test it on a few examples.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import re

def is_email(text):
    return re.match(r"^\S+@\S+\.\S+$", text) is not None

print(is_email("kamil@example.com"))
print(is_email("not_an_email"))
print(is_email("test@test"))
```
</details>

> 📝 **Exercise 2: Pulling all numbers out of text**
>
> Given `text = "Order #42 contains 3 items for a total of $129.99"`, use `re.findall()` to pull out every number (whole and decimal) as a list of strings.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import re

text = "Order #42 contains 3 items for a total of $129.99"
numbers = re.findall(r"\d+\.?\d*", text)
print(numbers)
```
</details>

> 📝 **Exercise 3: Normalizing whitespace**
>
> Given a string with multiple spaces/tabs in a row, e.g. `"This   is    text  with\t\tugly   spacing"`, use `re.sub()` to collapse every run of whitespace into a single space.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import re

text = "This   is    text  with\t\tugly   spacing"
cleaned = re.sub(r"\s+", " ", text)
print(cleaned)
```
</details>

> 📝 **Exercise 4: Masking a phone number**
>
> Given `data = "Contact: 600-100-200 or 22-555-4321"`, use `re.sub()` with groups to mask each number, keeping only the last 4 digits visible (e.g. `***-***-200`). Hint: use a simplified pattern that fits both formats, or handle them separately.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import re

data = "Contact: 600-100-200 or 22-555-4321"
masked = re.sub(r"\d{2,3}-\d{2,3}-(\d{3,4})", r"***-***-\1", data)
print(masked)
```
</details>

> 📝 **Exercise 5: Correctly reading CSV with a comma inside a value**
>
> Create a file `orders.csv` (with `csv.writer`) with columns `customer,items,total`, where the `items` column for one row contains a comma inside the value (e.g. `"bread, milk, eggs"` as a SINGLE value - pass it as one string in the list you give to `.writerow()`). Read the file back with `csv.reader` and show that this column comes back as one value, not three separate ones.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import csv

with open("orders.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["customer", "items", "total"])
    writer.writerow(["Kamil", "bread, milk, eggs", 25.5])

with open("orders.csv", "r", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    for row in reader:
        print(row)   # "items" is still one value, despite the comma inside it
```
</details>

> 📝 **Exercise 6: `DictReader` in practice**
>
> Read the `orders.csv` file from the previous exercise with `csv.DictReader` and, for each row, print a sentence like: "Kamil ordered: bread, milk, eggs for $25.5."

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import csv

with open("orders.csv", "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        print(f"{row['customer']} ordered: {row['items']} for ${row['total']}")
```
</details>

> 📝 **Exercise 7: Writing results with `DictWriter`**
>
> Given `results = [{'player': 'Kamil', 'score': 120}, {'player': 'Ania', 'score': 95}]`, write it to a file `results.csv` using `csv.DictWriter` (with a header), then read it back and print it to confirm it's correct.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import csv

results = [
    {"player": "Kamil", "score": 120},
    {"player": "Ania", "score": 95},
]

with open("results.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["player", "score"])
    writer.writeheader()
    writer.writerows(results)

with open("results.csv", "r", encoding="utf-8") as file:
    print(file.read())
```
</details>

> 🔥 **Exercise 8 (challenge): Server log parser**
>
> Create a file `server.log` (a few lines) in the format: `2026-03-15 10:22:01 192.168.1.10 GET /index.html 200` (date, time, IP, method, path, status code). Write a regex with groups that pulls out, from each line: the IP address, the HTTP method, and the status code. Count and print how many requests ended in an error status (`>= 400`) - treat the status as a number once you've extracted it from the group.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import re

with open("server.log", "w", encoding="utf-8") as file:
    file.write("2026-03-15 10:22:01 192.168.1.10 GET /index.html 200\n")
    file.write("2026-03-15 10:22:05 192.168.1.11 POST /login 401\n")
    file.write("2026-03-15 10:22:09 192.168.1.10 GET /missing.html 404\n")
    file.write("2026-03-15 10:22:12 192.168.1.12 GET /index.html 200\n")

pattern = r"(\d{1,3}(?:\.\d{1,3}){3}) (GET|POST|PUT|DELETE) \S+ (\d{3})"
error_count = 0

with open("server.log", "r", encoding="utf-8") as file:
    for line in file:
        match = re.search(pattern, line)
        if not match:
            continue
        ip, method, status = match.groups()
        status = int(status)
        print(f"IP: {ip}, method: {method}, status: {status}")
        if status >= 400:
            error_count += 1

print(f"Number of error requests: {error_count}")
```

Hint: `(?:...)` is a «non-capturing» group - it works like a regular group for repeating a piece of the pattern (`\.\d{1,3}` repeated 3 times for the IP address), but it doesn't clutter up the results from `.groups()`.
</details>

---

### What's next?

If the log parser went reasonably smoothly, you're ready for **Module 9: testing your
code** — how to automatically verify that your functions (including the ones you wrote
in this module) actually do what they're supposed to.